# Heston Model: From Calibration to Hedging

This notebook demonstrates a complete workflow for the Heston stochastic volatility model:
1. **Generate synthetic market data** with realistic parameters,
2. **Calibrate to implied vols** using vega-weighted least squares,
3. **Compute delta** using bump-and-revalue with common random numbers (CRN),
4. **Simulate hedging P&L** under the Heston model,
5. **Assess model risk** by hedging with Black-Scholes delta instead.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from lib.plot_style import colors

from py_vollib.black_scholes import black_scholes
from py_vollib.black_scholes.greeks.analytical import delta as bs_delta

from quantlab.data.synthetic import generate_heston_vol_surface
from quantlab.market_data.market_state import MarketState
from quantlab.models.heston.model import HestonParameters, HestonProcess

from quantlab.calibration.utils import calculate_vega_weights
from quantlab.calibration.utils import make_heston_object_wrapper
from quantlab.calibration.utils import safe_implied_volatility_of_undiscounted_price
from quantlab.calibration.inverse import recover_heston_params_from_implied_vols

from quantlab.pricing.heston.cos import price as cos_price
from quantlab.models.heston.closed_form import heston_call_price as analytic_price
from quantlab.sim.heston.mc_pricer import heston_euler_mc_price as mc_price

from quantlab.hedging.greeks import heston_delta_bump_revalue
from quantlab.instruments.base import StockOption

from quantlab.hedging.simulation import build_delta_interpolator, simulate_multiple_hedging_paths

## 1. Synthetic Market Generation

In [ ]:
# 1. Choose realistic Heston parameters (market-like)
target_params = {
    "v0": 0.04,      # 20% initial vol
    "kappa": 2.5,    # moderate mean reversion
    "theta": 0.04,   # 20% long-term vol
    "eta": 0.4,      # high vol-of-vol (creates smile)
    "rho": -0.6      # negative correlation (leverage effect)
}

# 2. Define market conditions
market_state = {"stock_price": 100.0, "interest_rate": 0.02, "time": 0.0}

# 3. Create realistic grid
strikes = np.linspace(70, 130, 13)  # ATM = 100
maturities = np.array([0.25, 0.5, 1.0, 1.5, 2.0])

# 4. Generate synthetic "market" implied vols (default output)
strikes_out, maturities_out, implied_vols_target = generate_heston_vol_surface(
    market_state=MarketState(**market_state),
    heston_params=HestonParameters(**target_params),
    strikes=strikes,
    maturities=maturities,
    output_format="implied_vols",  # Default
    pricing_method="cos",  # or "closed_form"
)
print(f"Generated {len(implied_vols_target)} synthetic prices.")

### Surface Visualization

We plot the volatility surface as:
1. **2D smiles**: Showing skew by maturity (standard analytical view),
2. **Contour heatmap**: Comprehensive surface view (industry standard for reporting).

The heatmap clearly shows the volatility smile intensity across strikes and maturities.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Smile by maturity
for T in np.unique(maturities_out):
    mask = maturities_out == T
    ax1.plot(strikes_out[mask], implied_vols_target[mask], 'o-', label=f'T={T:.2f}')
ax1.set_xlabel('Strike')
ax1.set_ylabel('Implied Volatility')
ax1.set_title('Volatility Smile (Synthetic Market)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Heatmap of surface
unique_strikes = np.unique(strikes_out)
unique_maturities = np.unique(maturities_out)
strike_mesh, mat_mesh = np.meshgrid(unique_strikes, unique_maturities)
ax2.contourf(strike_mesh, mat_mesh, implied_vols_target.reshape(len(unique_maturities), len(unique_strikes)))
ax2.set_xlabel('Strike')
ax2.set_ylabel('Maturity')
ax2.set_title('Volatility Surface (Synthetic Market)')
plt.colorbar(ax2.contourf(strike_mesh, mat_mesh, implied_vols_target.reshape(len(unique_maturities), len(unique_strikes))), ax=ax2)
plt.tight_layout()
plt.show()

## 2. Vega-Weighted Calibration
We calibrate the Heston model to a synthetic volatility surface.  
The objective function is vega-weighted MSE to reflect market quoting conventions:

$$
\mathcal{L}(\Theta) = \sum_{i} \nu_i^2 \left( \sigma^{\text{model}}_i - \sigma^{\text{market}}_i \right)^2
$$

where $\nu_i$ is the Black-Scholes vega for option $i$.

In [ ]:
# Calculate forwards and discount factors
S0, r = market_state["stock_price"], market_state["interest_rate"]
forwards = S0 * np.exp(r * maturities_out)
discount_factors = np.exp(-r * maturities_out)

# Calculate weights
vega_weights = calculate_vega_weights(
    strikes=strikes_out,
    maturities=maturities_out,
    implied_vols=implied_vols_target,
    forwards=forwards,
    interest_rates=np.full_like(maturities_out, market_state["interest_rate"])
)

In [ ]:
# Show how weights vary across the surface
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Weights vs Strike (by maturity)
for T_unique in np.unique(maturities_out):
    mask = maturities_out == T_unique
    ax1.scatter(strikes_out[mask], vega_weights[mask], 
               label=f'T={T_unique:.2f}', alpha=0.7)
ax1.set_xlabel('Strike')
ax1.set_ylabel('Vega Weight')
ax1.set_title('Vega Weights vs Strike')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Weights vs Maturity (by strike region)
for K_region in [80, 100, 120]:  # Select representative strikes
    mask = np.abs(strikes_out - K_region) < 5  # Within 5 of strike
    ax2.scatter(maturities_out[mask], vega_weights[mask], 
               label=f'K~{K_region}', alpha=0.7)
ax2.set_xlabel('Maturity')
ax2.set_ylabel('Vega Weight')
ax2.set_title('Vega Weights vs Maturity')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Weight statistics:")
print(f"  Mean: {vega_weights.mean():.6f}")
print(f"  Std:  {vega_weights.std():.6f}")
print(f"  Min:  {vega_weights.min():.6f}")
print(f"  Max:  {vega_weights.max():.6f}")

In [ ]:
cos_wrapper = make_heston_object_wrapper(
    pricer_func=cos_price,
    market_state_for_calibration=MarketState(**market_state),
    pricer_kwargs={"n_points": 4096},  # Use same settings as generation for fairness
)

# Use vega weights in calibration directly to implied vols
calibrated_params = recover_heston_params_from_implied_vols(
    strikes=strikes_out,
    maturities=maturities_out,
    target_implied_vols=implied_vols_target,
    market_state = MarketState(**market_state),
    initial_guess={"v0": 0.02, "kappa": 1.5, "theta": 0.1, "eta": 0.2, "rho": -0.4},
    pricing_func=cos_wrapper,
    pricing_kwargs={},
    bounds={
        "v0": (1e-4, 1.0),
        "kappa": (0.1, 20.0),
        "theta": (1e-4, 1.0),
        "eta": (0.01, 2.0),
        "rho": (-0.999, 0.999),
    },
    weights=vega_weights,  
    #method="L-BFGS-B",
    method="differential_evolution",
    optimizer_options={"maxiter": 100, "seed": 42, "polish":True, "disp": False},
    verbose=False
)

### Calibration Diagnostics

In [ ]:
# Recalculate fitted IVs using calibrated parameters
cos_wrapper_fitted = make_heston_object_wrapper(
    pricer_func=cos_price,
    market_state_for_calibration=MarketState(**market_state),
    pricer_kwargs={"n_points": 2048},
)

model_vols = []
for F, K, T, df in zip(forwards, strikes_out, maturities_out, discount_factors):
    # cos_wrapper returns discounted price
    model_price_discounted = cos_wrapper_fitted(calibrated_params, F, K, T)
    
    # Convert to undiscounted for IV calculation
    r = market_state["interest_rate"]
    T_expiry = T  
    undiscounted_price = model_price_discounted / np.exp(-r * T_expiry)
    
    # Calculate IV using undiscounted price
    model_iv = safe_implied_volatility_of_undiscounted_price(
        undiscounted_price, F, K, T_expiry, 'c'
    )
    model_vols.append(model_iv)

model_vols = np.array(model_vols)

# Plot 1: Target vs Fitted Surface
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Smile by maturity
for T_unique in np.unique(maturities_out):
    mask = maturities_out == T_unique
    ax1.scatter(strikes_out[mask], implied_vols_target[mask], 
               label=f'Target T={T_unique:.2f}', alpha=0.7, s=30)
    ax1.scatter(strikes_out[mask], model_vols[mask], 
               label=f'Fitted T={T_unique:.2f}', alpha=0.7, s=30, marker='x')
ax1.set_xlabel('Strike')
ax1.set_ylabel('Implied Volatility')
ax1.set_title('Calibration: Target vs Fitted Surface')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals vs Strike
residuals = implied_vols_target - model_vols
ax2.scatter(strikes_out, residuals, c=maturities_out, s=50, alpha=0.7)
ax2.axhline(0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('Strike')
ax2.set_ylabel('Residual (Target - Fitted)')
ax2.set_title('Calibration Residuals vs Strike')
ax2.grid(True, alpha=0.3)

# Residuals vs Maturity
ax3.scatter(maturities_out, residuals, c=strikes_out, s=50, alpha=0.7)
ax3.axhline(0, color='k', linestyle='--', alpha=0.5)
ax3.set_xlabel('Maturity')
ax3.set_ylabel('Residual (Target - Fitted)')
ax3.set_title('Calibration Residuals vs Maturity')
ax3.grid(True, alpha=0.3)

# Residuals histogram
ax4.hist(residuals, bins=30, alpha=0.7, edgecolor='black')
ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
ax4.set_xlabel('Residual (Target - Fitted)')
ax4.set_ylabel('Frequency')
ax4.set_title(f'Calibration Residual Distribution\n(RMS Error: {np.sqrt(np.mean(residuals**2)):.4f})')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare original vs calibrated parameters
comparison_data = {}
for param in target_params.keys():
    comparison_data[param] = {
        'True': target_params[param],
        'Calibrated': calibrated_params[param], 
        'Rel_Error (%)': abs(calibrated_params[param] - target_params[param]) / target_params[param] * 100
    }

comparison_df = pd.DataFrame(comparison_data).T  # Transpose to get params as rows

print("Parameter Recovery Comparison:")
print(comparison_df.round(4))

# Plot parameter bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Values
params = list(target_params.keys())
true_vals = [target_params[p] for p in params]
fitted_vals = [calibrated_params[p] for p in params]

x = np.arange(len(params))
width = 0.35

ax1.bar(x - width/2, true_vals, width, label='True', alpha=0.8)
ax1.bar(x + width/2, fitted_vals, width, label='Calibrated', alpha=0.8)
ax1.set_xlabel('Parameter')
ax1.set_ylabel('Value')
ax1.set_title('Parameter Recovery')
ax1.set_xticks(x)
ax1.set_xticklabels(params)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Relative errors
rel_errors = [(abs(calibrated_params[k] - target_params[k]) / target_params[k]) * 100 
              for k in target_params.keys()]

ax2.bar(params, rel_errors, alpha=0.7, color=colors['accent2'])
ax2.set_xlabel('Parameter')
ax2.set_ylabel('Relative Error (%)')
ax2.set_title('Parameter Recovery Error')
ax2.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive summary
metrics = {
    'RMS Error': np.sqrt(np.mean(residuals**2)),
    'Max Abs Error': np.max(np.abs(residuals)),
    'Mean Abs Error': np.mean(np.abs(residuals)),
    'R² Score': 1 - np.var(residuals) / np.var(implied_vols_target),
    'Calibration Success': 'Yes' if np.sqrt(np.mean(residuals**2)) < 0.01 else 'No'  # <1% IV error
}

print("\nCalibration Metrics:")
for metric, value in metrics.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.6f}")
    else:
        print(f"  {metric}: {value}")

# Impact of vega weighting
if vega_weights is not None:
    weighted_rms = np.sqrt(np.average(residuals**2, weights=vega_weights))
    unweighted_rms = np.sqrt(np.mean(residuals**2))
    print(f"\nImpact of Vega Weighting:")
    print(f"  Unweighted RMS Error: {unweighted_rms:.6f}")
    print(f"  Weighted RMS Error: {weighted_rms:.6f}")
    print(f"  Improvement: {((unweighted_rms - weighted_rms) / unweighted_rms * 100):.2f}%")

## 3. Delta Estimation

We compute delta using **bump-and-revalue with Common Random Numbers** to reduce Monte Carlo noise. This approach estimates:

$$
\Delta = \frac{\partial V}{\partial S_0} \approx \frac{V(S_0 + \epsilon) - V(S_0 - \epsilon)}{2\epsilon}
$$

where $V(S)$ is the option value under the Heston model, and common random numbers ensure correlated paths between $V(S_0+\epsilon)$ and $V(S_0-\epsilon)$.

In [ ]:
# Use the same parameters from calibration
market_state = MarketState(stock_price=100.0, interest_rate=0.02, time=0.0)
params = HestonParameters(**calibrated_params)
process = HestonProcess(params, market_state)

# Define a representative option
option = StockOption(strike_price=100.0, expiration_time=1.0, is_call=True)

# Compute Heston delta
delta_heston, price_0, price_up, price_down = heston_delta_bump_revalue(
    option,
    process,
    n_paths=100000,  # Can be adjusted for desired precision
    n_steps=252,     # Daily rebalancing 
    bump_size=0.01,  # 1% stock price bump
    seed=42
)

print(f"Heston Delta: {delta_heston:.4f}")
print(f"Option Price (S₀=100): {price_0:.4f}")
print(f"Price (S₀=101): {price_up:.4f}")
print(f"Price (S₀=99): {price_down:.4f}")
print(f"Finite difference: {(price_up - price_down) / (2 * 0.01):.4f}")

In [ ]:
# Calculate BS delta using ATM volatility from our surface
atm_strike = 100.0
atm_maturity = 1.0
atm_iv = np.interp(atm_strike, strikes_out[maturities_out == atm_maturity], 
                   implied_vols_target[maturities_out == atm_maturity])

# BS delta
delta_bs = bs_delta('c', 100.0, atm_strike, atm_maturity, 0.02, atm_iv)

print(f"Black-Scholes Delta (ATM IV): {delta_bs:.4f}")
print(f"Heston Delta:                {delta_heston:.4f}")
print(f"Difference:                  {abs(delta_heston - delta_bs):.4f}")

In [ ]:
# Compare delta across strikes
strikes_range = np.linspace(70, 130, 21)
deltas_heston = []
deltas_bs = []

for K in strikes_range:
    option_k = StockOption(strike_price=K, expiration_time=1.0, is_call=True)
    
    # Heston delta
    delta_h, _, _, _ = heston_delta_bump_revalue(
        option_k, process, n_paths=50000, n_steps=252, bump_size=0.01, seed=42
    )
    deltas_heston.append(delta_h)
    
    # Corresponding BS delta
    local_iv = np.interp(K, strikes_out[maturities_out == 1.0], 
                         implied_vols_target[maturities_out == 1.0])
    delta_bs_local = bs_delta('c', 100.0, K, 1.0, 0.02, local_iv)
    deltas_bs.append(delta_bs_local)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(strikes_range, deltas_heston, label='Heston Delta', linewidth=2)
plt.plot(strikes_range, deltas_bs, label='Black-Scholes Delta', linewidth=2)
plt.xlabel('Strike Price')
plt.ylabel('Delta')
plt.title('Delta Comparison: Heston vs Black-Scholes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Delta analysis complete. Heston accounts for stochastic volatility effects,")
print("leading to potentially different hedge ratios compared to BS.")

### Validation Note

Our Heston delta implementation has been validated against QuantLib using the same finite-difference approach. The bump-and-revalue method with Common Random Numbers provides:
- **Reduced Monte Carlo noise** through correlated path generation,
- **Accurate sensitivity estimates** for hedging purposes,
- **Consistency with industry-standard libraries** (QuantLib verification).

## 4. Hedging Performance

We simulate 1000 Heston paths, delta-hedge at 252 discrete times, and compute P&L.  
Hedging uses the **Heston delta** computed above.

In [ ]:
# Set up the hedging scenario
market_state = MarketState(stock_price=100.0, interest_rate=0.02, time=0.0)
params = HestonParameters(**calibrated_params)
process = HestonProcess(params, market_state)

# Define the option to hedge
option = StockOption(strike_price=100.0, expiration_time=1.0, is_call=True)

# Build delta interpolator grid
S_grid = np.linspace(70, 130, 21)  # Stock price grid
T_grid = np.linspace(0.01, 1.0, 21)  # Time to maturity grid

delta_interp = build_delta_interpolator(
    option=option,
    process=process,
    S_grid=S_grid,
    T_grid=T_grid,
    n_paths=10000,  # Reduce for notebook speed
    n_steps=100,
    seed=42
)

print("Delta interpolator built successfully.")

# Simulate hedging performance
n_paths = 1000
n_steps = 252  # Daily rebalancing for 1-year option

pnl_results = simulate_multiple_hedging_paths(
    option=option,
    process=process,
    initial_price=100.0,
    delta_interpolator=delta_interp,
    n_paths=n_paths,
    n_steps=n_steps,
    seed_start=1000
)

print(f"\nHedging Performance ({n_paths} paths, {n_steps} steps):")
print(f"Mean hedging P&L: ${pnl_results.mean():.4f}")
print(f"Std hedging P&L:  ${pnl_results.std():.4f}")
print(f"95% VaR:         ${np.percentile(pnl_results, 5):.4f}")
print(f"Max Loss:         ${pnl_results.min():.4f}")

In [ ]:
# Plot P&L distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(pnl_results, bins=50, alpha=0.7, density=True, edgecolor='black')
plt.axvline(pnl_results.mean(), color=colors['accent1'], linestyle='--', label=f'Mean P&L (${pnl_results.mean():.2f})')
plt.axvline(pnl_results.mean() + pnl_results.std(), color=colors['dark'], linestyle=':', alpha=0.7, label=f'+1σ (${(pnl_results.mean() + pnl_results.std()):.2f})')
plt.axvline(pnl_results.mean() - pnl_results.std(), color=colors['dark'], linestyle=':', alpha=0.7, label=f'-1σ (${(pnl_results.mean() - pnl_results.std()):.2f})')
plt.xlabel('Hedging P&L ($)')
plt.ylabel('Density')
plt.title('Distribution of Hedging P&L (Heston Delta)')
plt.legend()
plt.grid(True, alpha=0.3)


# Show key statistics
plt.subplot(1, 2, 2)
stats = [pnl_results.mean(), pnl_results.std(), np.percentile(pnl_results, 5)]
labels = ['Mean', 'Std', '5th Percentile']
bars = plt.bar(labels, stats, 
               color=[colors['primary'], colors['secondary'], colors['accent2']], 
               alpha=0.7)
plt.ylabel('P&L ($)')
plt.title('Hedging Performance Statistics')
plt.grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, stats):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'${val:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 5. Model Risk: Heston vs Black-Scholes Hedging

What happens when we hedge with the wrong model? We compare:
- **Heston delta hedging**: Using accurate stochastic volatility model
- **Black-Scholes delta hedging**: Using constant volatility approximation

This quantifies the **cost of model misspecification** in hedging strategies.

In [ ]:
### code

### Conclusion
- The Heston model captures volatility skew, leading to more stable hedging.
- Using BS delta under Heston dynamics increases P&L variance by ~X% — a quantifiable model risk.